In [1]:
import numpy as np
import pandas as pd

from utils import *
from astropy.cosmology import Planck18 as cosmo
import bilby
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display, clear_output
import corner
import json
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from matplotlib import rc
import os
#os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"

# Your custom text color
mycolor = 'black'  # or e.g., '#fe9463', (0.996, 0.580, 0.388)
from matplotlib import rcParams
# Global style settings
rcParams.update({
    # Enable LaTeX rendering
    #'text.usetex': True,

    # Font settings
    'font.family': 'sans-serif',
    'font.sans-serif': ['Open Sans'],  # Use Open Sans if available

    # Set all text and visual elements to use the custom color
    'text.color': mycolor,
    'axes.labelcolor': mycolor,
    'xtick.color': mycolor,
    'ytick.color': mycolor,
    'axes.edgecolor': mycolor,
    'grid.color': mycolor,
    'legend.edgecolor': mycolor,
    'legend.facecolor': 'none',
    'legend.labelcolor': mycolor,

    # Transparent backgrounds
    'figure.facecolor': 'none',
    'axes.facecolor': 'none',
})

if mycolor=='white':
    rcParams['savefig.transparent'] = True

import matplotlib as mpl
mpl.rcParams['figure.facecolor'] = 'none'
mpl.rcParams['axes.facecolor']   = 'none'
mpl.rcParams['savefig.facecolor'] = 'none'
# fix tick labels size
mpl.rcParams['xtick.labelsize'] = 15
mpl.rcParams['ytick.labelsize'] = 15

In [2]:
PATH_TO_20YR_CATALOG = 'Catalogue_global_ordered.txt'

In [3]:
injections_20years = pd.read_csv(PATH_TO_20YR_CATALOG,
                delimiter='\t', 
                names=['z', 'm1_source', 'm2_source', 'a1', 'a2', 'cos1', 'cos2', 'cos12', 'cluster'], skiprows=1)

In [4]:
injections_20years["cluster"] = injections_20years["cluster"].replace(
    "none", "isolated"
)

In [5]:
# check m1_source is always m2_source
(injections_20years['m1_source'] >= injections_20years['m2_source']).all()  

True

In [6]:
# filter for IMBH injections 
injections_imbh_20years = injections_20years[injections_20years['m1_source'] >= 100.]

In [7]:
injections_imbh_20years["cluster"].value_counts()

globular    12583
young        1440
nuclear       338
Name: cluster, dtype: int64

In [8]:
var_to_latex_dict = {
    'm1_source':r'\rm $m_1^{\rm src}\, [M_{\odot}]$',
    'm2_source':r'\rm $m_2^{\rm src}\, [M_{\odot}]$',
    'z':r'\rm $z$',
    'a1':r'\rm $a_1$',
    'a2':r'\rm $a_2$',
    'cos1':r'\rm $\cos(\theta_1)$',
    'cos2':r'\rm $\cos(\theta_2)$',
}
injection_plotter = make_injection_plotter(injections_imbh_20years, var_to_latex_dict)


In [9]:
def from_m1_m2_to_mChirp_q(m1, m2):
    """
    Compute the transformation from m1, m2 to mChirp, q
    """
    mChirp = (m1 * m2)**(3/5) / (m1 + m2)**(1/5)
    q = m2 / m1
    return mChirp, q

In [10]:
ns = len(injections_imbh_20years)
print('Number of IMBH observations in 20 years: ', ns)
print('Number of IMBH observations in 1 year: ', int(ns/20.))

Number of IMBH observations in 20 years:  14361
Number of IMBH observations in 1 year:  718


In [11]:
chirp_mass_20yr, mass_ratio_20yr = from_m1_m2_to_mChirp_q(injections_imbh_20years['m1_source'], injections_imbh_20years['m2_source'])
# fix the seed for reproducibility
np.random.seed(42)


# strating GPS time: Jan 1st 2042
start_time = 1956182418
seconds_in_a_year = 365.25 * 24 * 60 * 60

dec_data    = np.arccos(np.random.uniform(-1., 1., size = (ns,))) - np.pi / 2.
ra_data     = np.random.uniform(0, 2. * np.pi, size = (ns,))
theta_data  = np.arccos(np.random.uniform(-1., 1., size = (ns,)))
psi_data    = np.random.uniform(0, np.pi, size = (ns,))
time_data   = np.random.uniform(start_time, start_time + 20 * seconds_in_a_year, size = (ns,))  # 2042 --> 2062
phase_data  = np.random.uniform(0, 2. * np.pi, size = (ns,))

In [12]:
def get_bilby_spin_parametrization(a1, a2, cos1, cos2, cos12):
    tilt1 = np.arccos(cos1)
    tilt2 = np.arccos(cos2)
    phi12 = np.arccos(cos12)
    Jx = a1*np.sin(tilt1) + a2 * np.sin(tilt2) * cos12
    Jy = a2 * np.sin(tilt2) * np.sin(phi12)
    phijl = np.arctan2(Jy, Jx) % (2 * np.pi)

    return tilt1, tilt2, phi12, phijl
    

In [13]:
tilt_1, tilt_2, phi_12, phi_jl = get_bilby_spin_parametrization(injections_imbh_20years['a1'], injections_imbh_20years['a2'], injections_imbh_20years['cos1'], injections_imbh_20years['cos2'], injections_imbh_20years['cos12'])

In [14]:
# add the missing variables for the complete 15 parameters set
imbh_df_all = {
    'mass_1_source':injections_imbh_20years['m1_source'],
    'mass_2_source':injections_imbh_20years['m2_source'],
    'redshift':injections_imbh_20years['z'],
    'chirp_mass':chirp_mass_20yr * (1. + injections_imbh_20years['z']),
    'chirp_mass_source':chirp_mass_20yr,
    'mass_ratio':mass_ratio_20yr,
    'luminosity_distance':cosmo.luminosity_distance(injections_imbh_20years['z']),
    'dec':dec_data,
    'ra':ra_data,
    'theta_jn':theta_data,
    'psi':psi_data,
    'geocent_time':time_data,
    'phase':phase_data,
    'a_1':injections_imbh_20years['a1'],
    'a_2':injections_imbh_20years['a2'],
    'tilt_1':tilt_1,
    'tilt_2':tilt_2,
    'phi_12':phi_12,
    'phi_jl':phi_jl,
    'cluster':injections_imbh_20years['cluster']
}
imbh_df_data_all = pd.DataFrame(imbh_df_all).reset_index(drop=True)

In [15]:
min(imbh_df_all['mass_ratio'])

6.358325870830928e-05

In [16]:
imbh_df_var_to_latex = var_to_latex_dict = {
    'mass_1_source':r'\rm $m_1^{srm}\, [M_{\odot}]$',
    'mass_2_source':r'\rm $m_2^{srm}\, [M_{\odot}]$',
    'redshift':r'\rm $z$',
    'chirp_mass':r'\rm $\mathcal{M}_c\, [M_{\odot}]$',
    'chirp_mass_source':r'\rm $\mathcal{M}_c^{src}\, [M_{\odot}]$',
    'mass_ratio':r'\rm $q$',
    'luminosity_distance':r'\rm $d_L\, [\rm Mpc]$',
    'dec':r'\rm $\rm Dec\,[rad]$',
    'ra':r'\rm $\rm RA\,[rad]$',
    'theta_jn':r'\rm $\theta_{JN}\, [\rm rad]$',
    'psi':r'\rm $\psi\, [\rm rad]$',
    'phase':r'\rm $\ph\, [\rm rad]$',
    'a_1':r'\rm $a_1$',
    'a_2':r'\rm $a_2$',
    'tilt_1':r'\rm $\theta_1\, [\rm rad]$',
    'tilt_2':r'\rm $\theta_2\, [\rm rad]$',
    'phi_12':r'\rm $\phi_{12}\, [\rm rad]$',
    'phi_jl':r'\rm $\phi_{JL}\, [\rm rad]$',
}
injection_plotter = make_injection_plotter(imbh_df_data_all, imbh_df_var_to_latex)

In [17]:
imbh_df_data_all_cuts = imbh_df_data_all.copy(deep=True)
# change the values of the spin components that are exactly 0 to a small value (e.g. 1e-6) to avoid issues with the fisher matrix calculation
imbh_df_data_all_cuts['a_1'] = imbh_df_data_all_cuts['a_1'].replace(0, 1e-4)
imbh_df_data_all_cuts['a_2'] = imbh_df_data_all_cuts['a_2'].replace(0, 1e-4)
imbh_df_data_all_cuts.loc[
    imbh_df_data_all_cuts["mass_ratio"] < 1e-3,
    "mass_ratio"
] = 1e-2

In [18]:
imbh_df_data_all.to_hdf(f'injections/BPOP_imbh_injections_20years.hdf5', key='df', mode='w')

In [19]:
imbh_df_var_to_latex = var_to_latex_dict = {
    'mass_1_source':r'\rm $m_1^{srm}\, [M_{\odot}]$',
    'mass_2_source':r'\rm $m_2^{srm}\, [M_{\odot}]$',
    'redshift':r'\rm $z$',
    'chirp_mass':r'\rm $\mathcal{M}_c\, [M_{\odot}]$',
    'chirp_mass_source':r'\rm $\mathcal{M}_c^{src}\, [M_{\odot}]$',
    'mass_ratio':r'\rm $q$',
    'luminosity_distance':r'\rm $d_L\, [\rm Mpc]$',
    'dec':r'\rm $\rm Dec\,[rad]$',
    'ra':r'\rm $\rm RA\,[rad]$',
    'theta_jn':r'\rm $\theta_{JN}\, [\rm rad]$',
    'psi':r'\rm $\psi\, [\rm rad]$',
    'phase':r'\rm $\ph\, [\rm rad]$',
    'a_1':r'\rm $a_1$',
    'a_2':r'\rm $a_2$',
    'tilt_1':r'\rm $\theta_1\, [\rm rad]$',
    'tilt_2':r'\rm $\theta_2\, [\rm rad]$',
    'phi_12':r'\rm $\phi_{12}\, [\rm rad]$',
    'phi_jl':r'\rm $\phi_{JL}\, [\rm rad]$',
}
injection_plotter = make_injection_plotter(imbh_df_data_all_cuts, imbh_df_var_to_latex)

In [20]:
# read results 

delta_snr = pd.read_hdf('BPOP_imbh_injections_20years_SNR_ET_Delta.hdf5')
delta_snr

,mass_1_source,mass_2_source,redshift,chirp_mass,chirp_mass_source,mass_ratio,luminosity_distance,dec,ra,theta_jn,...,phase,a_1,a_2,tilt_1,tilt_2,phi_12,phi_jl,cluster,ET_in_band,SNR
0,107.984,28.4616,1.56930,118.958134,46.299822,0.263572,11832.079943,0.253630,4.964404,2.002523,...,2.948554,0.622,0.110,1.209486,1.095502,0.796239,0.107095,nuclear,True,56.705084
1,109.618,17.2568,1.10277,73.828149,35.109950,0.157427,7661.499175,-1.123058,4.144369,0.909880,...,1.697433,0.227,0.103,1.620817,0.373927,1.080599,0.134971,nuclear,True,62.564298
2,144.630,34.2461,0.87324,109.396811,58.399784,0.236784,5748.025590,-0.482492,5.382079,1.623246,...,3.792710,0.497,0.892,1.924101,2.896027,0.632419,0.197300,nuclear,True,90.999750
3,132.918,32.3787,0.65015,89.986324,54.532209,0.243599,4007.565639,-0.198620,0.475290,1.601727,...,2.769044,0.598,0.041,1.970125,2.056352,0.287697,0.017564,nuclear,True,167.854468
4,112.377,30.9116,3.34905,214.600843,49.344304,0.275071,29679.808217,0.758678,2.908051,1.121784,...,1.364160,0.624,0.073,0.939118,1.880525,0.574618,0.067144,nuclear,True,58.719434
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14356,648.225,67.3681,11.25010,2001.310391,163.370943,0.103927,121113.625416,-1.311284,6.107933,2.055674,...,0.015016,0.465,0.639,1.565596,2.087021,1.451917,0.804704,young,True,1.516097
14357,332.532,49.7579,10.43780,1183.262840,103.451961,0.149633,111274.446362,0.349400,0.273949,2.015357,...,2.245769,0.726,0.001,2.039803,2.409534,1.389607,0.001015,young,True,7.817417
14358,469.432,36.0619,10.32430,1123.158271,99.181254,0.076820,109904.882271,-0.852394,0.589544,1.304020,...,4.193701,0.170,0.001,1.561096,0.865972,0.348166,0.001522,young,True,3.125407
14359,244.802,22.2956,9.63730,607.740090,57.132928,0.091076,101644.505854,-0.756525,0.776833,1.263661,...,5.804382,0.258,0.001,1.026753,1.679209,1.712167,0.004461,young,True,14.605196


In [ ]:
injection_plotter = make_injection_plotter_detection(delta_snr, imbh_df_var_to_latex, in_band_column="ET_in_band", snr_column="SNR")